# 02 · C2 抗体：跑不了

论文 C2：抗体 CDRH3 设计，200 次评测后结合能均值比基线低 **18.2%**。

这条在本机起不来，卡在两个地方。


In [1]:
%matplotlib inline
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "font.size": 9,
    "axes.grid": True, "grid.alpha": .25,
    "axes.spines.top": False, "axes.spines.right": False,
})
R = Path("/lus/lfs1aip2/projects/public/u6gb/tasks/large-discovery-model/glm53_flash/results")

pd.DataFrame([
    ["Absolut! simulator", "external binary, not on this machine",
     "the whole C2 evaluation runs through it"],
    ["ldm-tts-antibody deps", "pins transformers 4.13 -> tokenizers 0.10.3",
     "no aarch64 wheel; needs a Rust build, already failed once (TEST_REPORT E3)"],
], columns=["blocker", "what it is", "why it stops C2"]).style.hide(axis="index")

blocker,what it is,why it stops C2
Absolut! simulator,"external binary, not on this machine",the whole C2 evaluation runs through it
ldm-tts-antibody deps,pins transformers 4.13 -> tokenizers 0.10.3,"no aarch64 wheel; needs a Rust build, already failed once (TEST_REPORT E3)"


## 说明

**Absolut!** 是 C2 的评测器 —— 抗体和抗原结合的结构模拟。它是个外部二进制，
得单独装，不在 pip 生态里。

**依赖那条更难办。** `ldm-tts-antibody` 把 transformers 钉在 4.13，
那个版本拉的是 tokenizers 0.10.3，而 0.10.3 在 aarch64 上没有预编译轮子。
装它就得现编 Rust，实测失败过一次（记在 TEST_REPORT E3）。

这不是「多花点时间就能装上」那种。老版本 tokenizers 的 Rust 代码
在 aarch64 上能不能编得过，取决于当年有没有人管这个平台 —— 大概率没有。

## 要跑的话，路有两条

**放宽依赖**：看 `ldm-tts-antibody` 是不是真的需要 transformers 4.13，
还是只是钉得太死。如果只用到 tokenizer 的基础功能，换新版可能就过了。
这个花半小时读一遍它的 import 就知道。

**换评测器**：Absolut! 装不上的话，C2 就没有可比的评测口径，
换别的模拟器等于换了实验，跟论文对不上。所以这条不通。

先做第一条 —— 依赖如果能松，Absolut! 那关才值得投入。